# GSPO (Group-wise Stochastic Policy Optimization)

基于组的随机策略优化方法

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

In [ ]:
class GSPOTrainer:
    def __init__(self, group_size=8, sample_size=4, use_baseline=True, temperature=1.0):
        self.group_size = group_size
        self.sample_size = sample_size
        self.use_baseline = use_baseline
        self.temperature = temperature
    
    def sample_from_group(self, group_rewards, group_log_probs, method='uniform'):
        if method == 'uniform':
            indices = np.random.choice(self.group_size, size=self.sample_size, replace=False)
        elif method == 'top_k':
            indices = np.argsort(group_rewards)[-self.sample_size:]
        else:
            indices = np.arange(self.sample_size)
        
        return group_rewards[indices], group_log_probs[indices], indices
    
    def gspo_loss(self, group_rewards, group_log_probs, num_samples=1):
        losses = []
        baseline = np.mean(group_rewards) if self.use_baseline else 0.0
        
        for _ in range(num_samples):
            sampled_rewards, sampled_log_probs, _ = self.sample_from_group(
                group_rewards, group_log_probs
            )
            advantages = (sampled_rewards - baseline) / self.temperature
            sample_loss = -np.mean(sampled_log_probs * advantages)
            losses.append(sample_loss)
        
        avg_loss = np.mean(losses)
        metrics = {'loss': avg_loss, 'loss_std': np.std(losses)}
        
        return avg_loss, metrics

In [ ]:
# 创建训练器
gspo = GSPOTrainer(group_size=8, sample_size=4, use_baseline=True)

# 模拟数据
group_rewards = np.array([0.9, 0.7, 0.5, 0.3, 0.8, 0.6, 0.4, 0.2])
group_log_probs = np.random.randn(8) * 0.5 - 2.5

# 计算损失
loss, metrics = gspo.gspo_loss(group_rewards, group_log_probs, num_samples=10)

print(f"损失: {loss:.4f}")
print(f"指标: {metrics}")

In [ ]:
# 可视化采样次数对方差的影响
num_samples_list = [1, 5, 10, 20, 50]
mean_losses = []
std_losses = []

for n_samples in num_samples_list:
    losses_exp = []
    for _ in range(100):
        loss_exp, _ = gspo.gspo_loss(group_rewards, group_log_probs, num_samples=n_samples)
        losses_exp.append(loss_exp)
    mean_losses.append(np.mean(losses_exp))
    std_losses.append(np.std(losses_exp))

plt.figure(figsize=(10, 6))
plt.errorbar(num_samples_list, mean_losses, yerr=std_losses, marker='o', capsize=5)
plt.xlabel('采样次数')
plt.ylabel('损失')
plt.title('采样次数对损失方差的影响')
plt.grid(True, alpha=0.3)
plt.show()

print("观察: 采样次数越多，方差越小")